# Training History Comparison Notebook

Use this notebook after training several models to compare their training curves and validation metrics.

It can read:

- Project training histories: `outputs/<model_name>/history.json`
- DDP histories: also saved as `outputs/<model_name>/history.json`
- YOLO11/Ultralytics histories: `runs/segment/train*/results.csv` or `outputs/yolo11_seg/results.csv`

The notebook creates:

- one table with available model histories,
- one comparison table with best epoch and best metric,
- training-loss comparison plots,
- validation-metric comparison plots,
- optional CSV export of the comparison summary.

The notebook does **not** retrain or modify any model.

## 1. Setup

Run this first. It locates the project root and imports the required packages.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import math
import sys
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# ---------------------------------------------------------------------
# Locate project root.
# This notebook is expected to be in:
#   radargram_hyperbola_benchmark/notebooks/training_history_comparison.ipynb
# but it also works if Jupyter starts from the project root.
# ---------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "src" / "radarseg").is_dir():
    candidate = PROJECT_ROOT.parent
    if (candidate / "src" / "radarseg").is_dir():
        PROJECT_ROOT = candidate

if not (PROJECT_ROOT / "src" / "radarseg").is_dir():
    raise RuntimeError(
        "Could not find the project root. Start Jupyter from the project root, "
        "or keep this notebook inside the project/notebooks folder."
    )

print(f"Project root: {PROJECT_ROOT}")

## 2. Choose model histories to compare

There are two ways to use this notebook.

### Option A — automatic discovery

Set `AUTO_DISCOVER = True`. The notebook searches common folders:

```text
outputs/*/history.json
runs/segment/train*/results.csv
outputs/yolo11_seg/results.csv
```

### Option B — manual selection

Set `AUTO_DISCOVER = False` and edit `MODEL_RUNS`.

Examples:

```python
MODEL_RUNS = {
    "U-Net": "outputs/unet/history.json",
    "UNet++": "outputs/unetpp/history.json",
    "Mask R-CNN": "outputs/mask_rcnn/history.json",
    "YOLO11-seg": "runs/segment/train/results.csv",
}
```

In [ ]:
# ---------------------------------------------------------------------
# User settings
# ---------------------------------------------------------------------

AUTO_DISCOVER = True

# Used only when AUTO_DISCOVER = False.
# Values can be a history.json file, a YOLO results.csv file, or a run folder.
MODEL_RUNS = {
    "U-Net": "outputs/unet/history.json",
    "UNet++": "outputs/unetpp/history.json",
    "SegFormer": "outputs/segformer/history.json",
    "DINOv3": "outputs/dinov3/history.json",
    "Mask R-CNN": "outputs/mask_rcnn/history.json",
    "Mask2Former": "outputs/mask2former/history.json",
    "YOLO11-seg": "runs/segment/train/results.csv",
}

# Main metrics to try, in priority order.
# The notebook will automatically keep only metrics that exist in the loaded histories.
PREFERRED_METRICS = [
    "train_loss",
    "dice",
    "iou",
    "semantic_dice",
    "semantic_iou",
    "instance_mean_matched_iou",
    "instance_ap50",
    "instance_precision",
    "instance_recall",
    "metrics/mAP50(M)",
    "metrics/mAP50-95(M)",
]

# Higher is better for these metric name fragments. Loss-like metrics are minimized.
HIGHER_IS_BETTER_FRAGMENTS = [
    "dice",
    "iou",
    "precision",
    "recall",
    "ap",
    "map",
    "accuracy",
]

## 3. Load histories

This cell defines robust loaders for the project JSON histories and YOLO CSV histories, then loads the selected runs.

In [ ]:
def resolve_path(path_like: str | Path) -> Path:
    """Resolve a path relative to PROJECT_ROOT unless it is already absolute."""
    path = Path(path_like).expanduser()
    if path.is_absolute():
        return path
    return PROJECT_ROOT / path


def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """Strip whitespace from CSV/JSON column names."""
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df


def load_project_history_json(path: Path) -> pd.DataFrame:
    """Load the project's history.json.

    Supported formats:
    1. List of epoch dictionaries:
       [{"epoch": 1, "train_loss": ...}, ...]

    2. Dictionary with a history key:
       {"history": [{"epoch": 1, ...}, ...]}

    3. Dictionary of metric lists:
       {"epoch": [1, 2], "train_loss": [0.5, 0.4], ...}
    """
    data = json.loads(path.read_text(encoding="utf-8"))

    if isinstance(data, dict) and "history" in data:
        data = data["history"]

    if isinstance(data, list):
        df = pd.DataFrame(data)
    elif isinstance(data, dict):
        # Handle dict-of-lists or dict-of-scalars.
        try:
            df = pd.DataFrame(data)
        except ValueError:
            df = pd.DataFrame([data])
    else:
        raise ValueError(f"Unsupported JSON history format in {path}: {type(data)}")

    if df.empty:
        raise ValueError(f"History file is empty: {path}")

    df = clean_column_names(df)
    if "epoch" not in df.columns:
        df.insert(0, "epoch", np.arange(1, len(df) + 1))

    return df


def load_yolo_results_csv(path: Path) -> pd.DataFrame:
    """Load Ultralytics YOLO results.csv.

    YOLO column names often include spaces, so they are stripped. If possible,
    a general train_loss column is created by summing the available train losses.
    """
    df = pd.read_csv(path)
    df = clean_column_names(df)

    if df.empty:
        raise ValueError(f"YOLO results.csv is empty: {path}")

    if "epoch" not in df.columns:
        df.insert(0, "epoch", np.arange(1, len(df) + 1))

    train_loss_cols = [
        c for c in df.columns
        if c.startswith("train/") and c.endswith("_loss")
    ]
    if train_loss_cols and "train_loss" not in df.columns:
        df["train_loss"] = df[train_loss_cols].sum(axis=1)

    return df


def find_history_file(run_path: str | Path) -> Path:
    """Resolve a history path from a file or run directory."""
    path = resolve_path(run_path)

    if path.is_file():
        return path

    if path.is_dir():
        candidates = [
            path / "history.json",
            path / "results.csv",
        ]
        candidates.extend(sorted(path.glob("**/history.json")))
        candidates.extend(sorted(path.glob("**/results.csv")))
        for candidate in candidates:
            if candidate.is_file():
                return candidate

    raise FileNotFoundError(f"No history.json or results.csv found for: {run_path}")


def infer_model_name_from_path(path: Path) -> str:
    """Infer a readable model name from a history path."""
    try:
        parts = path.relative_to(PROJECT_ROOT).parts
    except ValueError:
        parts = path.parts

    if "outputs" in parts:
        i = parts.index("outputs")
        if i + 1 < len(parts):
            return parts[i + 1]

    if "runs" in parts and "segment" in parts:
        # Example: runs/segment/train/results.csv
        return "yolo11_seg" if path.parent.name == "train" else f"yolo11_seg_{path.parent.name}"

    return path.parent.name


def discover_histories() -> dict[str, Path]:
    """Find available histories in common project folders."""
    discovered: dict[str, Path] = {}

    # Project histories.
    outputs_dir = PROJECT_ROOT / "outputs"
    if outputs_dir.is_dir():
        for path in sorted(outputs_dir.glob("*/history.json")):
            name = infer_model_name_from_path(path)
            discovered[name] = path

        # YOLO histories under project outputs.
        for path in sorted(outputs_dir.glob("yolo11_seg*/results.csv")):
            name = infer_model_name_from_path(path)
            discovered[name] = path

    # YOLO histories under Ultralytics default run folders.
    yolo_root = PROJECT_ROOT / "runs" / "segment"
    if yolo_root.is_dir():
        yolo_paths = sorted(yolo_root.glob("train*/results.csv"))
        for path in yolo_paths:
            run_name = path.parent.name
            name = "yolo11_seg" if run_name == "train" else f"yolo11_seg_{run_name}"
            discovered[name] = path

    return discovered


def load_history(path: Path) -> pd.DataFrame:
    """Load one history file based on extension/name."""
    if path.name == "history.json":
        return load_project_history_json(path)
    if path.name == "results.csv":
        return load_yolo_results_csv(path)

    if path.suffix.lower() == ".json":
        return load_project_history_json(path)
    if path.suffix.lower() == ".csv":
        return load_yolo_results_csv(path)

    raise ValueError(f"Unsupported history file type: {path}")


def to_numeric_history(df: pd.DataFrame) -> pd.DataFrame:
    """Convert metric-like columns to numeric safely.

    Some recent pandas versions no longer accept errors="ignore" in
    pd.to_numeric(...). This function uses errors="coerce" and only replaces a
    column when most non-empty values can actually be converted to numbers.
    """
    df = df.copy()
    for col in df.columns:
        if col in {"model", "history_path"}:
            continue

        series = df[col]
        if pd.api.types.is_numeric_dtype(series):
            continue

        converted = pd.to_numeric(series, errors="coerce")
        non_null = series.notna()

        if not non_null.any():
            df[col] = converted
            continue

        convertible_ratio = converted[non_null].notna().mean()

        # Training-history metric columns should be almost fully numeric. The
        # 0.8 threshold keeps occasional blanks safe while avoiding conversion
        # of text/object metadata columns.
        if convertible_ratio >= 0.8:
            df[col] = converted

    return df


if AUTO_DISCOVER:
    selected_paths = discover_histories()
else:
    selected_paths = {}
    manual_errors = {}
    for name, path in MODEL_RUNS.items():
        try:
            selected_paths[name] = find_history_file(path)
        except Exception as exc:
            manual_errors[name] = str(exc)

    if manual_errors:
        display(Markdown("### Manual paths not found"))
        display(pd.DataFrame([{"model": k, "error": v} for k, v in manual_errors.items()]))

if not selected_paths:
    display(Markdown("## No training histories were found"))
    display(Markdown(
        "The notebook looked for `outputs/*/history.json`, `runs/segment/train*/results.csv`, "
        "and `outputs/yolo11_seg*/results.csv`, but none were found.\\n\\n"
        "Check that you are running the notebook from the correct project root and that training has finished."
    ))
    raise FileNotFoundError(
        "No histories found. Set AUTO_DISCOVER=False and manually fill MODEL_RUNS with the correct paths."
    )

display(Markdown("### Candidate history files"))
display(pd.DataFrame([
    {
        "model": name,
        "path": str(path.relative_to(PROJECT_ROOT) if path.is_relative_to(PROJECT_ROOT) else path),
        "exists": path.exists(),
    }
    for name, path in selected_paths.items()
]).sort_values("model"))

histories: dict[str, pd.DataFrame] = {}
load_errors: dict[str, str] = {}

for name, path in selected_paths.items():
    try:
        df = load_history(path)
        df = to_numeric_history(df)
        df["model"] = name
        df["history_path"] = str(path.relative_to(PROJECT_ROOT) if path.is_relative_to(PROJECT_ROOT) else path)
        histories[name] = df
    except Exception as exc:
        load_errors[name] = str(exc)

loaded_rows = [
    {
        "model": name,
        "epochs": len(df),
        "history_path": df["history_path"].iloc[0],
        "columns": ", ".join([c for c in df.columns if c not in {"model", "history_path"}]),
    }
    for name, df in histories.items()
]

display(Markdown("### Loaded histories"))
if loaded_rows:
    loaded_table = pd.DataFrame(loaded_rows).sort_values("model").reset_index(drop=True)
    display(loaded_table)
else:
    loaded_table = pd.DataFrame(columns=["model", "epochs", "history_path", "columns"])
    display(Markdown("No history files were loaded successfully."))

if load_errors:
    display(Markdown("### Histories skipped because of errors"))
    display(pd.DataFrame([{"model": k, "error": v} for k, v in load_errors.items()]))

if not histories:
    raise RuntimeError(
        "No histories were loaded successfully. Review the candidate paths and errors above. "
        "If your JSON has a different format, inspect it with: !head -50 path/to/history.json"
    )

all_history = pd.concat(histories.values(), ignore_index=True)
print(f"Loaded {len(histories)} model training histories.")

## 4. Available metrics

This cell finds all numeric metrics that can be compared across models.

In [ ]:
def numeric_metric_columns(df: pd.DataFrame) -> list[str]:
    excluded = {"model", "history_path"}
    cols = []
    for col in df.columns:
        if col in excluded:
            continue
        if pd.api.types.is_numeric_dtype(df[col]):
            cols.append(col)
    return cols


def metric_direction(metric: str) -> str:
    """Return 'max' for higher-is-better metrics and 'min' for loss metrics."""
    metric_lower = metric.lower()
    if "loss" in metric_lower:
        return "min"
    for fragment in HIGHER_IS_BETTER_FRAGMENTS:
        if fragment in metric_lower:
            return "max"
    # Default to max for validation metrics unless clearly a loss.
    return "max"


available_metrics = numeric_metric_columns(all_history)
if not available_metrics:
    display(Markdown("## No numeric metrics found"))
    display(Markdown("The loaded history files did not contain numeric metric columns. Please inspect the loaded table above."))
    raise RuntimeError("No numeric metric columns found in loaded histories.")

preferred_available = [m for m in PREFERRED_METRICS if m in available_metrics]
other_metrics = [m for m in available_metrics if m not in preferred_available]
ordered_metrics = preferred_available + other_metrics

metrics_table = pd.DataFrame([
    {"metric": m, "direction": metric_direction(m), "available_for_models": int(all_history.groupby("model")[m].apply(lambda s: s.notna().any()).sum())}
    for m in ordered_metrics
])

display(metrics_table)

## 5. Best-epoch summary table

Choose one main metric for model ranking. For semantic models, `dice` or `iou` is usually useful.  
For instance models, `instance_mean_matched_iou` or `instance_ap50` is usually useful.

If the chosen metric is not available for a model, that model is skipped in the ranking table.

In [ ]:
# Choose the main metric for ranking models.
# Change this value after looking at the metrics table above.
MAIN_METRIC = (
    "instance_mean_matched_iou"
    if "instance_mean_matched_iou" in available_metrics
    else ("dice" if "dice" in available_metrics else ("train_loss" if "train_loss" in available_metrics else available_metrics[0]))
)

print("MAIN_METRIC:", MAIN_METRIC)


def best_row_for_metric(df: pd.DataFrame, metric: str) -> pd.Series | None:
    if metric not in df.columns:
        return None
    metric_values = pd.to_numeric(df[metric], errors="coerce")
    if metric_values.notna().sum() == 0:
        return None

    if metric_direction(metric) == "min":
        idx = metric_values.idxmin()
    else:
        idx = metric_values.idxmax()
    return df.loc[idx]


summary_rows = []
for name, df in histories.items():
    best = best_row_for_metric(df, MAIN_METRIC)
    if best is None:
        continue

    row = {
        "model": name,
        "best_epoch": int(best["epoch"]) if pd.notna(best["epoch"]) else None,
        f"best_{MAIN_METRIC}": float(best[MAIN_METRIC]),
        "history_path": best["history_path"],
    }

    # Add useful companion metrics if present.
    for metric in ["train_loss", "dice", "iou", "semantic_dice", "semantic_iou", "instance_mean_matched_iou", "instance_ap50", "instance_precision", "instance_recall", "metrics/mAP50(M)", "metrics/mAP50-95(M)"]:
        if metric in df.columns and metric not in row:
            value = best.get(metric, np.nan)
            if pd.notna(value):
                row[metric] = float(value)

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

if summary_df.empty:
    display(Markdown(f"No models have the selected metric `{MAIN_METRIC}`. Choose another metric."))
else:
    ascending = metric_direction(MAIN_METRIC) == "min"
    summary_df = summary_df.sort_values(f"best_{MAIN_METRIC}", ascending=ascending).reset_index(drop=True)
    display(Markdown(f"### Ranking by `{MAIN_METRIC}`"))
    display(summary_df)

## 6. Plot training curves

Each plot is created separately. Edit `METRICS_TO_PLOT` to control which metrics are shown.

In [ ]:
# Choose metrics to plot.
# Keep only metrics that exist in your histories.
METRICS_TO_PLOT = [
    m for m in [
        "train_loss",
        MAIN_METRIC,
        "dice",
        "iou",
        "semantic_dice",
        "instance_mean_matched_iou",
        "instance_ap50",
        "metrics/mAP50(M)",
        "metrics/mAP50-95(M)",
    ]
    if m in available_metrics
]

print("Metrics to plot:", METRICS_TO_PLOT)


def plot_metric(metric: str, histories: dict[str, pd.DataFrame]) -> None:
    """Plot one metric for all models that contain it."""
    plt.figure(figsize=(10, 6))

    plotted = 0
    for name, df in histories.items():
        if metric not in df.columns:
            continue
        values = pd.to_numeric(df[metric], errors="coerce")
        if values.notna().sum() == 0:
            continue

        epochs = pd.to_numeric(df["epoch"], errors="coerce")
        plt.plot(epochs, values, marker="o", linewidth=1.5, markersize=3, label=name)
        plotted += 1

    if plotted == 0:
        plt.close()
        print(f"No data available for metric: {metric}")
        return

    direction = metric_direction(metric)
    plt.title(f"{metric} over epochs ({'higher is better' if direction == 'max' else 'lower is better'})")
    plt.xlabel("Epoch")
    plt.ylabel(metric)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


for metric in METRICS_TO_PLOT:
    plot_metric(metric, histories)

## 7. Interactive metric selector

This optional cell provides dropdowns for changing the ranking metric and plot metric.  
If `ipywidgets` is not installed, use the manual cells above.

In [ ]:
try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

if widgets is None:
    display(Markdown("`ipywidgets` is not installed. Use the manual cells above."))
else:
    metric_dropdown = widgets.Dropdown(
        options=ordered_metrics,
        value=MAIN_METRIC if MAIN_METRIC in ordered_metrics else ordered_metrics[0],
        description="Metric:",
        layout=widgets.Layout(width="500px"),
    )

    output_area = widgets.Output()

    def update_metric(change=None):
        with output_area:
            output_area.clear_output()
            metric = metric_dropdown.value

            rows = []
            for name, df in histories.items():
                best = best_row_for_metric(df, metric)
                if best is None:
                    continue
                rows.append({
                    "model": name,
                    "best_epoch": int(best["epoch"]) if pd.notna(best["epoch"]) else None,
                    f"best_{metric}": float(best[metric]),
                    "history_path": best["history_path"],
                })

            if rows:
                result = pd.DataFrame(rows)
                result = result.sort_values(f"best_{metric}", ascending=(metric_direction(metric) == "min")).reset_index(drop=True)
                display(result)
            else:
                display(Markdown(f"No models contain metric `{metric}`."))

            plot_metric(metric, histories)

    metric_dropdown.observe(update_metric, names="value")
    display(widgets.VBox([metric_dropdown, output_area]))
    update_metric()

## 8. Export comparison summary and plots

This cell saves the ranking table and selected metric plots to:

```text
outputs/training_comparison/
```

In [ ]:
EXPORT_DIR = PROJECT_ROOT / "outputs" / "training_comparison"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

if "summary_df" in globals() and not summary_df.empty:
    summary_path = EXPORT_DIR / f"comparison_summary_by_{MAIN_METRIC.replace('/', '_')}.csv"
    summary_df.to_csv(summary_path, index=False)
    print(f"Saved summary table: {summary_path}")
else:
    print("No summary table to save.")

# Save one PNG per selected metric.
for metric in METRICS_TO_PLOT:
    safe_metric = metric.replace("/", "_").replace(" ", "_").replace("(", "").replace(")", "")
    out_path = EXPORT_DIR / f"{safe_metric}.png"

    plt.figure(figsize=(10, 6))
    plotted = 0
    for name, df in histories.items():
        if metric not in df.columns:
            continue
        values = pd.to_numeric(df[metric], errors="coerce")
        if values.notna().sum() == 0:
            continue
        epochs = pd.to_numeric(df["epoch"], errors="coerce")
        plt.plot(epochs, values, marker="o", linewidth=1.5, markersize=3, label=name)
        plotted += 1

    if plotted == 0:
        plt.close()
        continue

    plt.title(metric)
    plt.xlabel("Epoch")
    plt.ylabel(metric)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()
    print(f"Saved plot: {out_path}")